
# 🔥 EDA con FireDucks — versión completa

**FireDucks** acelera el código de **Pandas** sin cambiar su sintaxis, utilizando ejecución perezosa (lazy) y procesamiento paralelo.
Este notebook replica el mismo EDA que ya hicimos con **NumPy**, **Pandas**, **DuckDB** y **cuDF**, para comparar rendimiento y facilidad de uso.


## 0️⃣ Instalación y preparación del entorno

In [1]:

#!pip install -q fireducks

import time, numpy as np
import fireducks.pandas as pd

print("🔥 FireDucks activado. Versión:", pd.__version__)


🔥 FireDucks activado. Versión: 2.3.3


## 1️⃣ Creación del dataset sintético

In [2]:

np.random.seed(42)
N = 10_000_000  # tamaño grande para apreciar diferencias

ids = np.arange(1, N+1)
edad = np.random.randint(18, 65, size=N)
salario = np.random.normal(2500, 800, size=N)
experiencia = np.random.randint(0, 40, size=N)
horas = np.random.normal(40, 5, size=N)
nivel = np.random.randint(0, 5, size=N)
satisf = np.random.normal(7, 2, size=N)

# outliers (~0.5%)
out_idx = np.random.randint(0, N, size=N//200)
salario[out_idx] *= 4

# NaN (~1%)
nan_idx = np.random.randint(0, N, size=N//100)
satisf[nan_idx] = np.nan

df = pd.DataFrame({
    "ID": ids,
    "Edad": edad,
    "Salario": salario,
    "Experiencia": experiencia,
    "Horas": horas,
    "Nivel": nivel,
    "Satisfaccion": satisf
})

df.head()


,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfaccion
0,1,56,3233.679756,30,37.893934,3,10.377546
1,2,46,2539.954385,19,48.196550,0,7.065996
2,3,32,2789.993813,34,35.387557,2,3.219660
3,4,60,3103.731875,19,39.880112,2,10.391723
4,5,25,1893.981421,14,35.520172,3,6.668353


## 2️⃣ Descripción básica

In [3]:

t0 = time.time()
print("Filas x columnas:", df.shape)
print("Columnas:", list(df.columns))
display(df.head())
print("⏱️", time.time()-t0, "s")


Filas x columnas: (10000000, 7)
Columnas: ['ID', 'Edad', 'Salario', 'Experiencia', 'Horas', 'Nivel', 'Satisfaccion']


,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfaccion
0,1,56,3233.679756,30,37.893934,3,10.377546
1,2,46,2539.954385,19,48.196550,0,7.065996
2,3,32,2789.993813,34,35.387557,2,3.219660
3,4,60,3103.731875,19,39.880112,2,10.391723
4,5,25,1893.981421,14,35.520172,3,6.668353


⏱️ 0.007951021194458008 s


## 3️⃣ Tipos y rangos

In [4]:

t0 = time.time()
print(df.dtypes)
display(df[["Edad","Salario"]].describe())
print("⏱️", time.time()-t0, "s")


ID                int64
Edad              int64
Salario         float64
Experiencia       int64
Horas           float64
Nivel             int64
Satisfaccion    float64
dtype: object


,Edad,Salario
count,1.000000e+07,1.000000e+07
mean,4.100269e+01,2.537251e+03
std,1.356537e+01,9.835147e+02
min,1.800000e+01,-2.811805e+03
25%,2.900000e+01,1.962771e+03
50%,4.100000e+01,2.504663e+03
75%,5.300000e+01,3.048786e+03
max,6.400000e+01,2.293585e+04


⏱️ 1.771423578262329 s


## 4️⃣ Filtrado de datos

In [5]:

t0 = time.time()
altos = df[df["Salario"] > 4000]
cond2 = (df["Horas"] > 45) & (df["Experiencia"] > 10)
sel2 = df.loc[cond2, ["ID","Edad","Horas","Experiencia"]].head()
pct_nivel_ge3 = (df["Nivel"] >= 3).mean() * 100

print("Salarios >4000:", len(altos))
display(sel2)
print(f"% nivel ≥3: {pct_nivel_ge3:.2f}%")
print("⏱️", time.time()-t0, "s")


Salarios >4000: 351534


,ID,Edad,Horas,Experiencia
1,2,46,48.196550,19
17,18,19,45.192900,32
31,32,45,46.118014,21
38,39,24,46.084341,21
43,44,21,49.710731,16


% nivel ≥3: 40.01%
⏱️ 0.5094802379608154 s


## 5️⃣ Limpieza de valores faltantes

In [6]:

t0 = time.time()
nan_count = df["Satisfaccion"].isna().sum()
print("Valores nulos iniciales:", nan_count)
df["Satisfaccion"] = df["Satisfaccion"].fillna(df["Satisfaccion"].mean())
print("Nulos después:", df["Satisfaccion"].isna().sum())
print("⏱️", time.time()-t0, "s")


Valores nulos iniciales: 99494
Nulos después: 0
⏱️ 0.11989235877990723 s


## 6️⃣ Detección de outliers (IQR)

In [7]:

t0 = time.time()
q1 = df["Salario"].quantile(0.25)
q3 = df["Salario"].quantile(0.75)
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
filtro_out = (df["Salario"] < lim_inf) | (df["Salario"] > lim_sup)
print("Outliers detectados:", filtro_out.sum())
print("⏱️", time.time()-t0, "s")


Outliers detectados: 113567
⏱️ 1.4133832454681396 s


## 7️⃣ Sustitución de outliers (winsorizing)

In [9]:

t0 = time.time()
p95 = df["Salario"].quantile(0.95)
df.loc[filtro_out, "Salario"] = p95
print("Nuevo máximo salario:", df["Salario"].max())
print("⏱️", time.time()-t0, "s")


Nuevo máximo salario: 4677.799956231335
⏱️ 0.7260236740112305 s


## 8️⃣ Agrupaciones y resúmenes

In [10]:

t0 = time.time()
salario_media_nivel = df.groupby("Nivel")["Salario"].mean()
print("Salario medio por Nivel:")
display(salario_media_nivel)

bins = [0,30,50,200]
labels = ["<30","30–50",">50"]
df["GrupoEdad"] = pd.cut(df["Edad"], bins=bins, labels=labels)
print("Satisfacción media por grupo de edad:")
display(df.groupby("GrupoEdad")["Satisfaccion"].mean())
print("⏱️", time.time()-t0, "s")


Salario medio por Nivel:


Nivel
0    2515.388609
1    2515.899113
2    2516.094774
3    2514.507806
4    2516.805841
Name: Salario, dtype: float64

Satisfacción media por grupo de edad:


GrupoEdad
<30      7.000939
30–50    6.999378
>50      6.999760
Name: Satisfaccion, dtype: float64

⏱️ 2.65240216255188 s


## 9️⃣ Normalización y z-score

In [11]:

t0 = time.time()
df["Salario_norm"] = (df["Salario"] - df["Salario"].min()) / (df["Salario"].max() - df["Salario"].min())
df["Horas_z"] = (df["Horas"] - df["Horas"].mean()) / df["Horas"].std()
display(df[["Salario_norm","Horas_z"]].describe())
print("⏱️", time.time()-t0, "s")


,Salario_norm,Horas_z
count,1.000000e+07,1.000000e+07
mean,5.022938e-01,1.295575e-16
std,1.811471e-01,1.000000e+00
min,0.000000e+00,-5.894697e+00
25%,3.769532e-01,-6.746022e-01
50%,5.013116e-01,-2.468046e-04
75%,6.269920e-01,6.746371e-01
max,1.000000e+00,5.212721e+00


⏱️ 2.368612766265869 s


## 🔟 Correlaciones y resumen global

In [12]:

t0 = time.time()
print("Correlación salario–experiencia:")
display(df[["Salario","Experiencia"]].corr())
print("Correlación edad–satisfacción:")
display(df[["Edad","Satisfaccion"]].corr())
print("Resumen general:")
display(df.describe(include="all"))
print("⏱️", time.time()-t0, "s")


Correlación salario–experiencia:


,Salario,Experiencia
Salario,1.000000,0.000024
Experiencia,0.000024,1.000000


Correlación edad–satisfacción:


,Edad,Satisfaccion
Edad,1.000000,-0.000163
Satisfaccion,-0.000163,1.000000


Resumen general:


,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfaccion,GrupoEdad,Salario_norm,Horas_z
count,1.000000e+07,1.000000e+07,1.000000e+07,1.000000e+07,1.000000e+07,1.000000e+07,1.000000e+07,10000000,1.000000e+07,1.000000e+07
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30–50,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4255881,NaN,NaN
mean,5.000000e+06,4.100269e+01,2.515739e+03,1.950316e+01,3.999884e+01,2.000161e+00,6.999923e+00,NaN,5.022938e-01,1.295120e-16
std,2.886751e+06,1.356537e+01,7.869121e+02,1.154169e+01,5.002098e+00,1.414305e+00,1.989874e+00,NaN,1.811471e-01,1.000000e+00
min,1.000000e+00,1.800000e+01,3.337493e+02,0.000000e+00,1.051298e+01,0.000000e+00,-3.957017e+00,NaN,0.000000e+00,-5.894697e+00
25%,2.500001e+06,2.900000e+01,1.971253e+03,1.000000e+01,3.662441e+01,1.000000e+00,5.666968e+00,NaN,3.769532e-01,-6.746022e-01
50%,5.000000e+06,4.100000e+01,2.511472e+03,2.000000e+01,3.999760e+01,2.000000e+00,6.999923e+00,NaN,5.013116e-01,-2.468046e-04
75%,7.500000e+06,5.300000e+01,3.057434e+03,3.000000e+01,4.337344e+01,3.000000e+00,8.332475e+00,NaN,6.269920e-01,6.746371e-01


⏱️ 14.01847243309021 s


## 🚀 Comparativa de rendimiento FireDucks vs Pandas

In [14]:

import pandas as pd_std
pdf = df.to_pandas() if hasattr(df, "to_pandas") else df.copy()

def bench(func, name):
    t0 = time.time()
    func()
    print(f"{name:<30} ⏱️ {time.time()-t0:.3f}s")

print("🔥 Benchmark: FireDucks vs Pandas")
bench(lambda: df["Salario"].mean(), "FireDucks mean")
bench(lambda: pdf["Salario"].mean(), "Pandas mean")
bench(lambda: df.groupby("Nivel")["Salario"].mean(), "FireDucks groupby")
bench(lambda: pdf.groupby("Nivel")["Salario"].mean(), "Pandas groupby")


🔥 Benchmark: FireDucks vs Pandas
FireDucks mean                 ⏱️ 0.009s
Pandas mean                    ⏱️ 0.039s
FireDucks groupby              ⏱️ 0.000s
Pandas groupby                 ⏱️ 0.784s



## 🧠 Reflexión final

| Criterio | FireDucks | Pandas |
|-----------|------------|---------|
| API | Igual (totalmente compatible) | Nativo |
| Rendimiento | 🔥 Muy alto (lazy + paralelo) | Medio |
| Escalabilidad | Mejor en datasets medianos | Buena |
| Curva de aprendizaje | Nula | — |
| Soporte actual | En desarrollo activo | Maduro |
| Ideal para | Acelerar EDA y pipelines Pandas existentes | Análisis estándar |

**Conclusión:**  
FireDucks permite acelerar Pandas sin cambiar el código, con mejoras de entre 2× y 10× en operaciones típicas de EDA.
